In [ ]:
# 1) Importing Open AI and API Key
from openai import OpenAI
client = OpenAI(api_key="")

In [3]:
# 2) create a vector store
vector_store = client.vector_stores.create(name="visamate_store")

In [4]:
# 3) Uploading every TXT file and linking it to the store
txt_files = [
    "Ireland_Student_Visa_Info.txt",
    "Ireland_Tourist_Visa_Info.txt",
    "Schengen_Student_Visa_Info.txt",
    "Schengen_Tourist_Visa_Info.txt",
    "UK_Student_Visa_Info.txt",
    "UK_Tourist_Visa_Info.txt",
]

for path in txt_files:
    fid = client.files.create(
        file=open(path, "rb"),
        purpose="assistants",
    ).id
    client.vector_stores.files.create(
        vector_store_id = vector_store.id,
        file_id         = fid,
    )

In [5]:
# 4) Poll until all files are indexed
while True:
    status = client.vector_stores.retrieve(vector_store.id).status
    if status == "completed":
        break
    print("Indexing …", status)
    time.sleep(2)

In [6]:
# 5) build an assistant that points file_search at that store
assistant = client.beta.assistants.create(
    name="VisaMate",
    model="gpt-3.5-turbo",
    instructions=(
        "You are VisaMate, a visa advisor for Ireland, Schengen and the UK. "
        "Always ground answers in the vector-store knowledge base and use "
        "code when date or currency calculations are needed."
    ),
    tools=[{"type": "file_search"}, {"type": "code_interpreter"}],
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

In [ ]:
# 6) VisaMate Interactive Chat Loop
import time
from openai._exceptions import RateLimitError

def visamate_chat(assistant_id):

    # Start a new thread so the assistant can maintain conversation context
    thread = client.beta.threads.create()
    print("VisaMate Assistant is ready. Ask your visa questions below.")
    print("Type 'exit' or 'quit' to end the chat.")

    while True:
        user_input = input("\nYou: ").strip()

        # Check if the user wants to end the chat
        if user_input.lower() in ("exit", "quit"):
            print("Session ended. Thank you!")
            break

        # Add the user's input to the thread
        client.beta.threads.messages.create(
            thread_id = thread.id,
            role      = "user",
            content   = user_input
        )

        # Request the assistant to generate a response
        while True:
            try:
                run = client.beta.threads.runs.create(
                    thread_id    = thread.id,
                    assistant_id = assistant_id
                )
                break
            except RateLimitError:
                print("Rate limit reached. Retrying in 30 seconds...")
                time.sleep(30)

        # Wait until the assistant finishes generating the response
        while True:
            run = client.beta.threads.runs.retrieve(
                thread_id = thread.id,
                run_id    = run.id
            )

            # If the assistant is trying to use a tool, handle it here
            if run.status == "requires_action":
                tool_calls   = run.required_action.submit_tool_outputs.tool_calls
                tool_outputs = []

                # Currently no custom tools; we proceed with empty outputs
                run = client.beta.threads.runs.submit_tool_outputs(
                    thread_id    = thread.id,
                    run_id       = run.id,
                    tool_outputs = tool_outputs
                )
                continue

            if run.status in ("completed", "failed"):
                break

            time.sleep(1.5)

        # Display the result
        if run.status == "failed":
            print("Error: The assistant encountered a problem:", run.last_error) # Error handling
        else:
            reply_msg = client.beta.threads.messages.list(
                thread_id = thread.id,
                order     = "desc",
                limit     = 1
            ).data[0]
            answer = reply_msg.content[0].text.value.strip()
            print("\nVisaMate:", answer)

# Start the chat
visamate_chat(assistant.id)

VisaMate Assistant is ready. Ask your visa questions below.
Type 'exit' or 'quit' to end the chat.
